# GRIT 0.5B on Google Colab (one A100 80GB)

This notebook runs the pinned `Qwen/Qwen2.5-0.5B-Instruct` policy with the pinned `Qwen/Qwen3Guard-Gen-0.6B` safety reward. It keeps the same task corpus, 1,000 projector contexts, 6,000 KL contexts, top-64 plus tail trust region, GRPO grouping, projected AdamW predictor, and central-difference correction as `main.ipynb`.

Choose **Runtime → Change runtime type → A100 80GB GPU**. The actor and persistent vLLM process share GPU 0; this Colab profile is restricted to one actor and the 0.5B policy. Preparation and training use the same GPU at different times. Generated data and complete checkpoints are kept on Google Drive. Colab GPU time and Drive storage are required; the full 40-step duration is unmeasured.

Run cells in order. The preparation and training cells may take a long time. Do not put credentials in the notebook or repository.


## 1. Mount Drive and provide the repository source

Set `REPO_URL` to your GitHub clone URL **after pushing this notebook and code changes**. If the repo is not on GitHub, leave it empty and place a source-only copy at `DRIVE_REPO`. Keep generated data out of that source folder. A private GitHub repo must be accessible to the Colab runtime without embedding a token in the URL.


In [1]:
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, subprocess, sys

drive.mount('/content/drive')
REPO_URL = 'https://github.com/namdeptraivcd/GRIT.git'  # e.g. https://github.com/your-name/GRIT.git
DRIVE_REPO = Path('/content/drive/MyDrive/GRIT-source')
REPO = Path('/content/GRIT')
if not REPO.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
    else:
        assert (DRIVE_REPO / 'scripts/train_grit.py').is_file(), 'Set REPO_URL or upload the source folder to DRIVE_REPO.'
        shutil.copytree(DRIVE_REPO, REPO, ignore=shutil.ignore_patterns('.git', 'data', 'artifacts', 'checkpoints', '__pycache__'))
assert (REPO / 'grit/model_profiles.py').is_file(), 'Source copy is stale; sync the new small-model support.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source:', REPO)


Mounted at /content/drive
Source: /content/GRIT


## 2. Install the same PyTorch/vLLM stack and check the GPU

Use a fresh Colab GPU runtime. This installs the repository's Linux/CUDA dependencies; if Colab asks for a runtime restart, restart and rerun the setup cells before preparation.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-vllm.txt'], check=True)
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 1, 'Select a one-GPU A100 80GB runtime.'
properties = torch.cuda.get_device_properties(0)
assert 'A100' in properties.name and properties.total_memory >= 75_000_000_000, f'Need A100 80GB; got {properties.name}, {properties.total_memory} bytes.'
print(properties.name, round(properties.total_memory / 1e9, 1), 'GB')


## 3. Configure persistent storage and Hub access

Create `HF_TOKEN` in **Colab Secrets**. Give it permission to create/write the private checkpoint repo. The next cell prepares all datasets automatically, including 1,000 held-out PKU-SafeRLHF evaluation prompts at `GRIT_colab_0p5b/eval/evaluation_prompts.parquet`; no data upload is needed. Evaluation prompts exclude task train, projector and KL prompts. Keep enough Drive space for the 7,000 generated contexts and repeated FP32 checkpoints. With `save_steps=2`, twenty full checkpoints can require roughly 120 GB just for 0.5B FP32 weights and AdamW state; check your Drive quota before training.


In [ ]:
from grit.model_profiles import SMALL
from scripts.preservation_data import preservation_paths

DRIVE_ROOT = Path('/content/drive/MyDrive/GRIT_colab_0p5b')
DATA_ROOT = DRIVE_ROOT / 'data'
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
OUTPUT_ROOT = DRIVE_ROOT / 'checkpoints'
EVALUATION_FILE = DRIVE_ROOT / 'eval/evaluation_prompts.parquet'
HUB_REPO_ID = ''  # e.g. your-name/grit-qwen25-0p5b
RUN_ID = 'qwen25-0p5b-grit-v1'  # Keep fixed across Colab reconnects for resume.

HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Add HF_TOKEN to Colab Secrets and grant notebook access.'
assert HUB_REPO_ID, 'Set HUB_REPO_ID.'
os.environ['HF_TOKEN'] = HF_TOKEN
for directory in (DATA_ROOT, ARTIFACT_ROOT, OUTPUT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
paths = preservation_paths(DATA_ROOT, ARTIFACT_ROOT, SMALL.policy)
TASK_FILE = DATA_ROOT / 'task/task_train.parquet'
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
print({'policy': SMALL.policy, 'reward': SMALL.safety, 'data': str(DATA_ROOT), 'run': str(OUTPUT_DIR)})


## 4. Automatically prepare task, evaluation and preservation data

Prepare task data, sample projector/KL prompts, then reserve 1,000 unique evaluation prompts from unused PKU-SafeRLHF source rows (seed 66), excluding normalized overlaps with all three corpora. This held-out safety corpus is used for overlap checks, not a general/math/code benchmark. Existing evaluation files are validated and reused without being overwritten. Then generate the 7,000 frozen-base contexts and build projectors for the pinned 0.5B model. Complete artifacts are reused; incomplete attempts are kept for inspection. GPU generation can take a long time; this cell does **not** start training.


In [ ]:
if not TASK_FILE.is_file():
    subprocess.run([sys.executable, '-u', 'scripts/prepare_grit_data.py', '--task-only', '--output-dir', str(TASK_FILE.parent)], check=True)
prep_env = dict(os.environ, DATA_ROOT=str(DATA_ROOT), ARTIFACT_ROOT=str(ARTIFACT_ROOT),
                MODEL_PATH=SMALL.policy, MODEL_REVISION=SMALL.policy_revision,
                PROJECTORS_PATH=paths['projectors_path'], PYTHON_BIN=sys.executable,
                PYTHONPATH=str(REPO))
print('Sampling projector/KL prompts...', flush=True)
subprocess.run(['bash', 'scripts/run_preservation_projectors.sh', 'sample'], env=prep_env, check=True)
prompt_root = DATA_ROOT / 'preservation/split_v1/prompts'
print('Preparing and validating held-out evaluation prompts...', flush=True)
subprocess.run([sys.executable, '-u', 'scripts/prepare_evaluation_data.py',
                '--task-file', str(TASK_FILE), '--output', str(EVALUATION_FILE),
                '--exclude-prompts', str(prompt_root / 'projector/preserve_prompts.parquet'),
                str(prompt_root / 'monitor/preserve_prompts.parquet')], check=True)
for stage in ('generate', 'build'):
    print(f'Preservation stage: {stage}', flush=True)
    subprocess.run(['bash', 'scripts/run_preservation_projectors.sh', stage], env=prep_env, check=True)
for path in (TASK_FILE, EVALUATION_FILE, Path(paths['preserve_file']), Path(paths['projectors_path'])):
    assert path.is_file(), path
print('Preparation complete:', paths)


## 5. Verify CPU invariants and configure training

The global batch, optimizer, KL support, curvature settings and frozen data match `main.ipynb`. One actor handles all 321 prompts per step. `rollout_memory_utilization=0.2` reserves a smaller GPU share for vLLM because it shares the A100 with the actor; this needs a real GPU fit check.


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'test_function/test_training.py', 'test_function/test_notebook.py'], check=True)
config = {
    'task_file': str(TASK_FILE),
    'preserve_file': paths['preserve_file'],
    'evaluation_file': str(EVALUATION_FILE),
    'projectors_path': paths['projectors_path'],
    'output_dir': str(OUTPUT_DIR),
    'model_path': SMALL.policy,
    'model_revision': SMALL.policy_revision,
    'safety_model_path': SMALL.safety,
    'safety_model_revision': SMALL.safety_revision,
    'rollout_gpu': '0',
    'allow_colocated_rollout': True,
    'rollout_memory_utilization': 0.2,
    'rollout_dtype': 'auto',
    'task_optimizer': 'adamw',
    'projector_relaxation': 0.05,
    'task_batch_size': 321,
    'preserve_batch_size': 48,
    'generations': 5,
    'max_prompt_length': 512,
    'max_response_length': 256,
    'max_preserve_length': 2304,
    'lr': 1e-6,
    'lambda_pres': 1.0,
    'epsilon_pres': 1e-3,
    'top_k': 64,
    'base_statistics_dtype': 'float16',
    'use_curvature': True,
    'fd_radius': 0.05,
    'fd_check_interval': 10,
    'gradient_topk': 5,
    'gradient_checkpointing': True,
    'max_steps': 40,
    'save_steps': 2,
    'hub_repo_id': HUB_REPO_ID,
    'seed': 66,
}
from scripts.train_grit import parse_args
arguments = []
for key, value in config.items():
    flag = '--' + key.replace('_', '-')
    if isinstance(value, bool):
        if value:
            arguments.append(flag)
    elif value is not None:
        arguments.extend([flag, str(value)])
parse_args(arguments)
print('Training configuration validated; output:', OUTPUT_DIR)


## 6. Train or resume from the latest complete Drive checkpoint

This cell starts the GPU training job. Checkpoints are saved every two steps. If Colab ends, reconnect, rerun the setup cells, and execute this cell again; it selects the latest checkpoint with `complete.json`. Keep `RUN_ID` and config unchanged. An interrupted checkpoint directory is renamed and retained so the next save can use its step number. If a run folder exists without any complete checkpoint, inspect it and use a new `RUN_ID`; no data is deleted automatically.


In [ ]:
from datetime import datetime, timezone
if OUTPUT_DIR.exists():
    for candidate in sorted(OUTPUT_DIR.glob('step_*')):
        if candidate.is_dir() and not (candidate / 'complete.json').is_file():
            stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
            archived = OUTPUT_DIR / f'interrupted_{candidate.name}_{stamp}'
            candidate.rename(archived)
            print('Preserved interrupted checkpoint:', archived)
completed = sorted(OUTPUT_DIR.glob('step_*/complete.json'),
                   key=lambda path: int(path.parent.name.split('_')[-1])) if OUTPUT_DIR.exists() else []
train_args = arguments.copy()
if completed:
    last = completed[-1].parent
    last_step = int(last.name.split('_')[-1])
    if last_step >= config['max_steps']:
        print('Training already complete at step', last_step)
    else:
        train_args.extend(['--resume', str(last)])
        print('Resuming from', last)
elif OUTPUT_DIR.exists():
    raise RuntimeError(f'Incomplete run folder without a complete checkpoint: {OUTPUT_DIR}')
if not completed or int(completed[-1].parent.name.split('_')[-1]) < config['max_steps']:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', PYTHONPATH=str(REPO), OMP_NUM_THREADS='4')
    subprocess.run([sys.executable, 'scripts/train_grit.py', *train_args], env=env, check=True)


## 7. Inspect persisted results

`metrics.jsonl`, `diagnostics_summary.json`, complete checkpoints and optimizer state stay in Drive. The first step records KL at the frozen base against the stored top-64 statistics.


In [ ]:
import json
metrics_path = OUTPUT_DIR / 'metrics.jsonl'
if metrics_path.is_file():
    metrics = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    print('Completed steps:', len(metrics))
    print(json.dumps(metrics[-1], indent=2) if metrics else 'No completed step yet')
summary_path = OUTPUT_DIR / 'diagnostics_summary.json'
if summary_path.is_file():
    print('Diagnostics:', summary_path.read_text())
